# Partial observability on CartPole

CartPole’s full observation has four features: cart position, cart velocity, pole angle, pole velocity. You can **mask** indices so the agent only sees a subset — here position and angle, hiding velocities.

## `observation_indices`

`observation_indices` is a list of integer indices to keep from a continuous-vector observation. It applies to the raw Gymnasium observation before any other formatting, so the resulting observation tensor has shape `(len(observation_indices),)` instead of the full observation shape.

```python
cfg = EnvConfig(
    id="CartPole-v1",
    seed=0,
    num_envs=4,
    max_episode_steps=500,
    observation_indices=[0, 2],  # keep: cart_pos, pole_angle
)
```

`env.output_specs[0].observation.shape` reflects the masked shape — `(2,)` here rather than `(4,)`. The contract is always described over the masked observation, so network input sizes and replay buffer allocations can read it directly without knowing the original shape.

**When is this useful?** Partial observability experiments test how much information an agent actually needs. Removing velocity features from CartPole simulates a sensor that measures position and angle but not their derivatives — a harder control problem that forces the agent to infer velocity from history.

In [ ]:
import numpy as np
from mouse_envs import EnvConfig, make_env

# CartPole obs indices: 0=cart_pos, 1=cart_vel, 2=pole_angle, 3=pole_vel
VISIBLE_INDICES = [0, 2]

## Config

`observation_indices` masks the Gymnasium observation before mouse-env formats it.

In [ ]:
cfg = EnvConfig(
    id="CartPole-v1",
    seed=0,
    num_envs=2,
    episodes_per_task=100,
    observation_indices=VISIBLE_INDICES,
)
env = make_env(cfg)

## Verify observation width

In [ ]:
outputs = env.step(env.sample_random_inputs())
batch_obs = np.stack([r["observation"].numpy() for r in outputs])
print(f"obs shape: {batch_obs.shape}")  # (2, 2)

## Rollout (shape stays `(num_envs, 2)`)

In [ ]:
for _ in range(200):
    outputs = env.step(env.sample_random_inputs())

batch_obs = np.stack([r["observation"].numpy() for r in outputs])
print(f"final obs shape: {batch_obs.shape}")
env.close()